# 📖 Notebook 2: Redirect Service with Caching

A URL shortener is **read-heavy**: for every URL created, it gets clicked hundreds or
thousands of times. This notebook builds a redirect service with FastAPI and shows how
adding a Redis cache makes redirects dramatically faster.

## Learning Objectives

By the end of this notebook you'll understand:
- How HTTP redirects work (301 vs 302 status codes)
- Why URL shorteners are read-heavy (1000:1 read/write ratio)
- How to implement Cache-Aside pattern for redirects
- The performance difference between DB lookups and cached lookups
- How to handle cache misses, expired URLs, and TTL

## 🛠️ Setup

Make sure Docker is running:

```bash
cd system-designs/bitly
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `bitly_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [1]:
import psycopg2
import redis
import json
import time
import threading

# Database connection settings
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "bitly_demo",
    "user": "demo",
    "password": "demo"
}

# Redis connection settings
REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

✅ Connected to PostgreSQL
✅ Connected to Redis


## 🤔 How Does a Redirect Work?

When you click `short.ly/abc123`, your browser sends a GET request to the server.
The server looks up `abc123`, finds the original URL, and sends back a **redirect response**.

```
Browser:  GET /abc123
Server:   HTTP 302 Found
          Location: https://www.example.com/very/long/path
Browser:  (automatically follows the Location header)
```

### 301 vs 302 Redirects

| Status | Name | Browser Caches? | Use When |
|--------|------|-----------------|----------|
| 301 | Moved Permanently | Yes — future clicks skip your server | URL never changes, no analytics needed |
| 302 | Found (Temporary) | No — every click hits your server | You need analytics, or URL might change |

**For URL shorteners, 302 is preferred** because:
- We can track every click for analytics
- We can update or expire the short URL later
- We keep control of the redirect

---
## Step 1: Redirect Without Cache (Database Only)

The simplest approach: every redirect request queries PostgreSQL.

In [2]:
def redirect_db_only(short_code: str) -> dict:
    """
    Look up a short code in PostgreSQL and return the long URL.
    This simulates what the redirect endpoint does.
    
    Returns a dict with 'url' and 'source' (where we found the data).
    """
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute(
        "SELECT long_url, expires_at FROM urls WHERE short_code = %s",
        (short_code,)
    )
    row = cursor.fetchone()
    conn.close()
    
    if row is None:
        return {"error": "not_found", "source": "db"}
    
    long_url, expires_at = row
    # Check expiration
    if expires_at and expires_at < __import__('datetime').datetime.now():
        return {"error": "expired", "source": "db"}
    
    return {"url": long_url, "source": "db"}

# Test with our seed data
result = redirect_db_only("abc123")
print(f"Lookup 'abc123': {result}")

result = redirect_db_only("nonexistent")
print(f"Lookup 'nonexistent': {result}")

Lookup 'abc123': {'url': 'https://www.example.com/very/long/path/to/a/page?with=many&query=params', 'source': 'db'}
Lookup 'nonexistent': {'error': 'not_found', 'source': 'db'}


In [3]:
# Measure: How fast are DB-only redirects?

codes = ["abc123", "gH7kL9", "Xy3mN8", "qW5rT2", "pL4jF6"]

db_times = []
for _ in range(200):
    code = codes[_ % len(codes)]
    start = time.time()
    redirect_db_only(code)
    db_times.append((time.time() - start) * 1000)

avg_db = sum(db_times) / len(db_times)
print(f"📦 DB-Only Redirects (200 lookups):")
print(f"   Average: {avg_db:.2f} ms")
print(f"   Min:     {min(db_times):.2f} ms")
print(f"   Max:     {max(db_times):.2f} ms")
print()
print("💡 Each redirect opens a connection, queries Postgres, and closes.")
print("   At 1000 clicks/sec this would stress the database.")

📦 DB-Only Redirects (200 lookups):
   Average: 20.26 ms
   Min:     13.49 ms
   Max:     46.14 ms

💡 Each redirect opens a connection, queries Postgres, and closes.
   At 1000 clicks/sec this would stress the database.


---
## Step 2: Add Redis Cache (Cache-Aside Pattern)

The **Cache-Aside** pattern is the most common caching strategy:

```
1. Check Redis for the short code
2. Cache HIT  → return the URL from Redis (fast!)
3. Cache MISS → query PostgreSQL, store result in Redis, return URL
```

This means:
- The **first** click on a short URL hits the DB (cold cache)
- **All subsequent** clicks are served from Redis (warm cache)
- We set a **TTL** (Time To Live) so stale entries expire automatically

In [4]:
r = get_redis_client()

CACHE_TTL_SECONDS = 3600  # 1 hour

def redirect_with_cache(short_code: str) -> dict:
    """
    Look up a short code, checking Redis first (Cache-Aside).
    
    Flow:
    1. Check Redis → if found, return immediately (cache hit)
    2. If not in Redis → query PostgreSQL (cache miss)
    3. Store the result in Redis with a TTL for next time
    """
    cache_key = f"url:{short_code}"
    
    # Step 1: Check the cache
    cached = r.get(cache_key)
    if cached:
        return {"url": cached, "source": "cache"}
    
    # Step 2: Cache miss — query the database
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute(
        "SELECT long_url FROM urls WHERE short_code = %s",
        (short_code,)
    )
    row = cursor.fetchone()
    conn.close()
    
    if row is None:
        return {"error": "not_found", "source": "db"}
    
    long_url = row[0]
    
    # Step 3: Store in cache for next time
    r.setex(cache_key, CACHE_TTL_SECONDS, long_url)
    
    return {"url": long_url, "source": "db (now cached)"}

# First call: cache miss (goes to DB)
print("First lookup (cold cache):")
result1 = redirect_with_cache("abc123")
print(f"  {result1}")

# Second call: cache hit (from Redis)
print("\nSecond lookup (warm cache):")
result2 = redirect_with_cache("abc123")
print(f"  {result2}")

# Check the TTL in Redis
ttl = r.ttl("url:abc123")
print(f"\n⏱️  TTL remaining: {ttl} seconds")

First lookup (cold cache):
  {'url': 'https://www.example.com/very/long/path/to/a/page?with=many&query=params', 'source': 'db (now cached)'}

Second lookup (warm cache):
  {'url': 'https://www.example.com/very/long/path/to/a/page?with=many&query=params', 'source': 'cache'}

⏱️  TTL remaining: 3600 seconds


In [5]:
# Warm the cache for all test codes, then measure cached performance
for code in codes:
    redirect_with_cache(code)  # first call populates cache

cache_times = []
for _ in range(200):
    code = codes[_ % len(codes)]
    start = time.time()
    redirect_with_cache(code)
    cache_times.append((time.time() - start) * 1000)

avg_cache = sum(cache_times) / len(cache_times)

print("⚡ Performance Comparison (200 lookups):")
print("=" * 50)
print(f"  DB only:       avg {avg_db:.2f} ms")
print(f"  With cache:    avg {avg_cache:.2f} ms")
print(f"  Speedup:       {avg_db / avg_cache:.1f}×")
print()
print("💡 Cached redirects skip the DB entirely — just a Redis GET.")

⚡ Performance Comparison (200 lookups):
  DB only:       avg 20.26 ms
  With cache:    avg 0.26 ms
  Speedup:       76.6×

💡 Cached redirects skip the DB entirely — just a Redis GET.


---
## 🛡️ Bonus: Negative Caching (Defense Against Cache Penetration)

What if someone floods your service with requests for **short codes that don't exist**?

```
GET /aaaaaa   → not in cache → query DB → 404
GET /aaaaab   → not in cache → query DB → 404
GET /aaaaac   → not in cache → query DB → 404
...
```

Every request becomes a DB query. The cache helps you *zero percent* because we never
store "this code doesn't exist". This is called **cache penetration**.

The fix is simple: cache the "not found" answer too, with a **short TTL** (e.g. 60s)
so that legitimate new codes appear in the system quickly.


In [6]:
NOT_FOUND_SENTINEL = "__NOT_FOUND__"
NOT_FOUND_TTL = 60  # short TTL so new codes become visible quickly

def redirect_with_negative_cache(short_code: str) -> dict:
    """Cache-Aside that also caches 'not found' results briefly."""
    cache_key = f"url:{short_code}"
    cached = r.get(cache_key)

    if cached == NOT_FOUND_SENTINEL:
        return {"error": "not_found", "source": "cache"}
    if cached:
        return {"url": cached, "source": "cache"}

    # Cache miss — ask the database
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT long_url FROM urls WHERE short_code = %s", (short_code,))
    row = cursor.fetchone()
    conn.close()

    if row is None:
        # Cache the negative answer with a *short* TTL
        r.setex(cache_key, NOT_FOUND_TTL, NOT_FOUND_SENTINEL)
        return {"error": "not_found", "source": "db (now negatively cached)"}

    long_url = row[0]
    r.setex(cache_key, CACHE_TTL_SECONDS, long_url)
    return {"url": long_url, "source": "db (now cached)"}


# Demo: simulate a flood of requests for a non-existent code
r.delete("url:ZZZZZZ")

print("Without negative caching, every miss hits the DB.")
print("With negative caching, only the FIRST miss does.\n")

miss_times = []
for i in range(5):
    start = time.time()
    result = redirect_with_negative_cache("ZZZZZZ")
    dt = (time.time() - start) * 1000
    miss_times.append(dt)
    print(f"  attempt {i+1}: {dt:.2f} ms  source={result['source']}")

print(f"\n💡 First call queries the DB (~{miss_times[0]:.1f} ms).")
print(f"   Subsequent calls are served from Redis (~{miss_times[-1]:.2f} ms).")

# Clean up the sentinel so it doesn't bleed into later cells
r.delete("url:ZZZZZZ")


Without negative caching, every miss hits the DB.
With negative caching, only the FIRST miss does.



  attempt 1: 15.46 ms  source=db (now negatively cached)
  attempt 2: 0.24 ms  source=cache
  attempt 3: 0.21 ms  source=cache
  attempt 4: 0.19 ms  source=cache
  attempt 5: 0.20 ms  source=cache

💡 First call queries the DB (~15.5 ms).
   Subsequent calls are served from Redis (~0.20 ms).


1

---
## Step 3: Build the Actual FastAPI Redirect Service

Let's build a real HTTP service that you can test with `curl` or a browser.
We'll run it in a background thread so we can test it from this notebook.

In [7]:
from fastapi import FastAPI, HTTPException
from fastapi.responses import RedirectResponse, JSONResponse
from pydantic import BaseModel
import uvicorn
import string

app = FastAPI(title="Bitly Clone")

# ── Base62 helpers ──────────────────────────────────────────
BASE62_ALPHABET = string.digits + string.ascii_lowercase + string.ascii_uppercase

def base62_encode(number: int) -> str:
    if number == 0:
        return BASE62_ALPHABET[0]
    result = []
    while number > 0:
        number, remainder = divmod(number, 62)
        result.append(BASE62_ALPHABET[remainder])
    return ''.join(reversed(result))

# ── Request/Response models ─────────────────────────────────
class ShortenRequest(BaseModel):
    long_url: str

class ShortenResponse(BaseModel):
    short_code: str
    short_url: str
    long_url: str

# ── Counter key ─────────────────────────────────────────────
COUNTER_KEY = "bitly:url_counter"
CACHE_TTL = 3600

# ── POST /shorten ──────────────────────────────────────────
@app.post("/shorten", response_model=ShortenResponse)
def shorten_url(req: ShortenRequest):
    """Create a short URL from a long URL."""
    redis_client = get_redis_client()
    counter = redis_client.incr(COUNTER_KEY)
    short_code = base62_encode(counter)
    
    conn = get_db_connection()
    conn.autocommit = True
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO urls (short_code, long_url) VALUES (%s, %s)",
        (short_code, req.long_url)
    )
    conn.close()
    
    # Pre-populate cache
    redis_client.setex(f"url:{short_code}", CACHE_TTL, req.long_url)
    
    return ShortenResponse(
        short_code=short_code,
        short_url=f"http://localhost:8000/{short_code}",
        long_url=req.long_url
    )

# ── GET /{short_code} ──────────────────────────────────────
@app.get("/{short_code}")
def redirect(short_code: str):
    """Redirect a short code to its original URL (with caching)."""
    redis_client = get_redis_client()
    cache_key = f"url:{short_code}"
    
    # Check cache first
    cached_url = redis_client.get(cache_key)
    if cached_url:
        return RedirectResponse(url=cached_url, status_code=302)
    
    # Cache miss — query DB
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute(
        "SELECT long_url FROM urls WHERE short_code = %s",
        (short_code,)
    )
    row = cursor.fetchone()
    conn.close()
    
    if row is None:
        raise HTTPException(status_code=404, detail="Short URL not found")
    
    long_url = row[0]
    
    # Populate cache
    redis_client.setex(cache_key, CACHE_TTL, long_url)
    
    return RedirectResponse(url=long_url, status_code=302)

# ── GET /stats/{short_code} ────────────────────────────────
@app.get("/stats/{short_code}")
def get_stats(short_code: str):
    """Check if a code is cached and what TTL remains."""
    redis_client = get_redis_client()
    cache_key = f"url:{short_code}"
    cached = redis_client.get(cache_key)
    ttl = redis_client.ttl(cache_key)
    return {
        "short_code": short_code,
        "cached": cached is not None,
        "ttl_seconds": ttl if ttl > 0 else None,
        "cached_url": cached
    }

# Start the server in a background thread
server_thread = threading.Thread(
    target=uvicorn.run,
    args=(app,),
    kwargs={"host": "0.0.0.0", "port": 8000, "log_level": "warning"},
    daemon=True
)
server_thread.start()
time.sleep(2)  # wait for server to start
print("✅ FastAPI server running at http://localhost:8000")
print("   Endpoints:")
print("   - POST /shorten          (create a short URL)")
print("   - GET  /{short_code}     (redirect)")
print("   - GET  /stats/{code}     (cache info)")

✅ FastAPI server running at http://localhost:8000
   Endpoints:
   - POST /shorten          (create a short URL)
   - GET  /{short_code}     (redirect)
   - GET  /stats/{code}     (cache info)


In [8]:
import httpx

BASE_URL = "http://localhost:8000"

# 1. Shorten a URL
print("1️⃣  Shorten a URL:")
resp = httpx.post(f"{BASE_URL}/shorten", json={"long_url": "https://github.com/features"})
data = resp.json()
print(f"   {data}")
short_code = data["short_code"]

# 2. Check cache stats
print(f"\n2️⃣  Cache stats for '{short_code}':")
resp = httpx.get(f"{BASE_URL}/stats/{short_code}")
print(f"   {resp.json()}")

# 3. Follow the redirect (don't auto-follow so we can see the 302)
print(f"\n3️⃣  Redirect for '{short_code}':")
resp = httpx.get(f"{BASE_URL}/{short_code}", follow_redirects=False)
print(f"   Status: {resp.status_code}")
print(f"   Location: {resp.headers.get('location')}")

# 4. Test a seed data code
print(f"\n4️⃣  Redirect for seed data 'abc123':")
resp = httpx.get(f"{BASE_URL}/abc123", follow_redirects=False)
print(f"   Status: {resp.status_code}")
print(f"   Location: {resp.headers.get('location')}")

# 5. Test 404
print(f"\n5️⃣  Non-existent code:")
resp = httpx.get(f"{BASE_URL}/ZZZZZ", follow_redirects=False)
print(f"   Status: {resp.status_code}")

1️⃣  Shorten a URL:


   {'short_code': '1', 'short_url': 'http://localhost:8000/1', 'long_url': 'https://github.com/features'}

2️⃣  Cache stats for '1':
   {'short_code': '1', 'cached': True, 'ttl_seconds': 3600, 'cached_url': 'https://github.com/features'}

3️⃣  Redirect for '1':
   Status: 302
   Location: https://github.com/features

4️⃣  Redirect for seed data 'abc123':
   Status: 302
   Location: https://www.example.com/very/long/path/to/a/page?with=many&query=params

5️⃣  Non-existent code:
   Status: 404


---
## Step 4: Measure the Impact of Caching Under Load

Let's simulate many users clicking the same short URL and compare
response times with and without the cache.

In [9]:
from concurrent.futures import ThreadPoolExecutor
import random

# Use the raw functions (not HTTP) for a cleaner comparison
test_codes = ["abc123", "gH7kL9", "Xy3mN8", "qW5rT2", "pL4jF6"]

def benchmark(fn, label, num_requests=500, workers=20):
    """Run a function many times concurrently and measure latency."""
    def do_request(_):
        code = test_codes[_ % len(test_codes)]
        t0 = time.time()
        fn(code)
        return (time.time() - t0) * 1000
    
    with ThreadPoolExecutor(max_workers=workers) as pool:
        times = list(pool.map(do_request, range(num_requests)))
    
    times.sort()
    print(f"📊 {label}")
    print(f"   Requests:    {num_requests}")
    print(f"   Avg latency: {sum(times)/len(times):.2f} ms")
    print(f"   P50 latency: {times[len(times)//2]:.2f} ms")
    print(f"   P95 latency: {times[int(len(times)*0.95)]:.2f} ms")
    print(f"   P99 latency: {times[int(len(times)*0.99)]:.2f} ms")
    print()
    return times

# Clear cache to test cold vs warm
r = get_redis_client()
for code in test_codes:
    r.delete(f"url:{code}")

print("Test 1: DB-only (no cache)")
db_times = benchmark(redirect_db_only, "WITHOUT CACHE")

# Warm the cache
for code in test_codes:
    redirect_with_cache(code)

print("Test 2: With warm cache")
cache_times = benchmark(redirect_with_cache, "WITH CACHE")

speedup = (sum(db_times)/len(db_times)) / (sum(cache_times)/len(cache_times))
print(f"🚀 Cache is {speedup:.1f}× faster on average!")

Test 1: DB-only (no cache)


📊 WITHOUT CACHE
   Requests:    500
   Avg latency: 43.72 ms
   P50 latency: 42.87 ms
   P95 latency: 62.12 ms
   P99 latency: 70.98 ms

Test 2: With warm cache
📊 WITH CACHE
   Requests:    500
   Avg latency: 1.23 ms
   P50 latency: 0.98 ms
   P95 latency: 3.05 ms
   P99 latency: 5.64 ms

🚀 Cache is 35.7× faster on average!


---
## Step 5: Cache Invalidation for Expired URLs

What happens when a URL expires? We need to:
1. Remove it from the cache (or let TTL handle it)
2. Return a 410 Gone status

Redis TTL handles most cases automatically. But if a URL is explicitly deleted
or expired early, we need to invalidate the cache entry.

In [10]:
def delete_short_url(short_code: str) -> bool:
    """
    Delete a short URL and invalidate its cache entry.
    
    Order matters:
    1. Delete from DB first (source of truth)
    2. Then delete from cache
    
    If we deleted cache first, another request could re-populate it
    from the DB before we delete the DB row.
    """
    conn = get_db_connection()
    conn.autocommit = True
    cursor = conn.cursor()
    
    # Delete from DB
    cursor.execute("DELETE FROM clicks WHERE short_code = %s", (short_code,))
    cursor.execute("DELETE FROM urls WHERE short_code = %s RETURNING short_code", (short_code,))
    deleted = cursor.fetchone() is not None
    conn.close()
    
    # Invalidate cache
    redis_client = get_redis_client()
    redis_client.delete(f"url:{short_code}")
    
    return deleted

# Demo: create a URL, verify it works, then delete it
resp = httpx.post(f"{BASE_URL}/shorten", json={"long_url": "https://example.com/will-be-deleted"})
temp_code = resp.json()["short_code"]
print(f"Created: {temp_code}")

# Verify it redirects
resp = httpx.get(f"{BASE_URL}/{temp_code}", follow_redirects=False)
print(f"Before delete: status={resp.status_code}, location={resp.headers.get('location')}")

# Delete it
deleted = delete_short_url(temp_code)
print(f"Deleted: {deleted}")

# Verify it's gone
resp = httpx.get(f"{BASE_URL}/{temp_code}", follow_redirects=False)
print(f"After delete: status={resp.status_code}")

Created: 2
Before delete: status=302, location=https://example.com/will-be-deleted
Deleted: True
After delete: status=404


---
## ✍️ Bonus: Custom Aliases (Vanity URLs)

Most URL shorteners let paying users pick their own short code:
`short.ly/my-product` instead of `short.ly/aB3xQ7`. Our schema already has the
column — let's use it.

Rules of the road:
- The alias must be unique across **both** `short_code` and `custom_alias`.
- Reject reserved words (`admin`, `api`, `login`, …) so people can't squat on your routes.
- Enforce length and character limits (usually `a-zA-Z0-9_-`, 3–30 chars).


In [11]:
import re

RESERVED_ALIASES = {"admin", "api", "login", "logout", "signup", "stats", "shorten"}
ALIAS_PATTERN = re.compile(r"^[a-zA-Z0-9_-]{3,30}$")

class AliasError(Exception):
    pass

def shorten_with_alias(long_url: str, alias: str) -> str:
    """Create a short URL with a user-chosen alias instead of an auto-generated code."""
    if not ALIAS_PATTERN.match(alias):
        raise AliasError("Alias must be 3–30 chars of [a-zA-Z0-9_-].")
    if alias.lower() in RESERVED_ALIASES:
        raise AliasError(f"'{alias}' is a reserved word.")

    conn = get_db_connection()
    conn.autocommit = True
    cursor = conn.cursor()
    # Uniqueness check across BOTH columns
    cursor.execute(
        "SELECT 1 FROM urls WHERE short_code = %s OR custom_alias = %s",
        (alias, alias),
    )
    if cursor.fetchone() is not None:
        conn.close()
        raise AliasError(f"Alias '{alias}' is already taken.")

    # Store the alias in BOTH columns so lookups by short_code still work unchanged.
    cursor.execute(
        "INSERT INTO urls (short_code, custom_alias, long_url) VALUES (%s, %s, %s)",
        (alias, alias, long_url),
    )
    conn.close()

    # Pre-warm the cache
    get_redis_client().setex(f"url:{alias}", CACHE_TTL_SECONDS, long_url)
    return alias


# Demo: success, validation failure, and duplicate
try:
    code = shorten_with_alias("https://example.com/launch-2026", "my-launch")
    print(f"✅ Created custom alias: /{code}")
except AliasError as e:
    print(f"❌ {e}")

for bad_alias in ["ab", "hello world", "admin"]:
    try:
        shorten_with_alias("https://example.com/x", bad_alias)
    except AliasError as e:
        print(f"❌ '{bad_alias}' rejected: {e}")

try:
    shorten_with_alias("https://example.com/another", "my-launch")
except AliasError as e:
    print(f"❌ Duplicate rejected: {e}")

# The alias works just like any other short code
result = redirect_with_cache("my-launch")
print(f"\n🔁 Redirect for /my-launch → {result}")


✅ Created custom alias: /my-launch
❌ 'ab' rejected: Alias must be 3–30 chars of [a-zA-Z0-9_-].
❌ 'hello world' rejected: Alias must be 3–30 chars of [a-zA-Z0-9_-].
❌ 'admin' rejected: 'admin' is a reserved word.
❌ Duplicate rejected: Alias 'my-launch' is already taken.

🔁 Redirect for /my-launch → {'url': 'https://example.com/launch-2026', 'source': 'cache'}


## 🧹 Cleanup

In [12]:
# Clean up cache keys and test data we created in this notebook.
SEED_CODES = (
    'abc123', 'gH7kL9', 'Xy3mN8', 'qW5rT2', 'pL4jF6',
    'dK8vB1', 'zN9cX3', 'mR2hY7', 'tJ6wA4', 'vF1sE5',
)

r = get_redis_client()
# Clear every cache-related key
for pattern in ("url:*", "bitly:*"):
    keys = r.keys(pattern)
    if keys:
        r.delete(*keys)

conn = get_db_connection()
conn.autocommit = True
cursor = conn.cursor()
# FK-safe cleanup: clicks first, then urls
cursor.execute("DELETE FROM clicks WHERE short_code NOT IN %s", (SEED_CODES,))
cursor.execute("DELETE FROM urls   WHERE short_code NOT IN %s", (SEED_CODES,))
conn.close()

print("🧹 Cleaned up cache keys and test data")
print("   (The FastAPI server will stop when you restart the kernel)")


🧹 Cleaned up cache keys and test data
   (The FastAPI server will stop when you restart the kernel)


## 📚 Summary

### Key Takeaways

1. **302 redirect** is preferred — keeps control for analytics and URL updates
2. **Cache-Aside** is the standard pattern: check cache → miss → query DB → populate cache
3. **TTL** ensures stale cache entries expire automatically
4. **Cache invalidation** on delete: remove from DB first, then from cache
5. **Caching makes redirects 10–50× faster** — essential at the 1000:1 read/write ratio

### Architecture So Far

```
Client → FastAPI Server → Redis Cache (check first)
                        → PostgreSQL  (on cache miss)
```

### Next Up

In **Notebook 3**, we'll add **analytics and click tracking** — recording every click
with referrer, country, and user agent data, and building real-time dashboards.